# Analyse et Traitement des Données Pédologiques HWSD2

Ce notebook combine :
- L'analyse exploratoire du dataset HWSD2
- Le diagnostic et nettoyage des valeurs manquantes
- Le traitement des valeurs aberrantes et l'agrégation spatiale
- L'encodage des variables catégorielles

## Partie 1: Analyse Exploratoire (AnalyseS2)

### Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

### 📂 Chargement des Données

In [ ]:
print("="*70)
print("📂 CHARGEMENT DES DONNÉES")
print("="*70)

df = pd.read_csv(
    r"C:\Users\Y A N I S\Desktop\set\output\HWSD2_Algeria_Tunisia_D1_CLEAN.csv",
    sep=";"
)

print(f"✅ Dataset chargé : {len(df):,} lignes × {len(df.columns)} colonnes")
print(f"📊 Mémoire utilisée : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

### 🔧 Preprocessing - Ajout de Colonnes

In [ ]:
print("\n" + "="*70)
print("🔧 PREPROCESSING - AJOUT DE NOUVELLES COLONNES")
print("="*70)
print("⚠️  Le dataset ORIGINAL reste intact (fichier CSV non modifié)")
print("⚠️  On ajoute juste des colonnes EN MÉMOIRE pour l'analyse\n")

# Avant preprocessing
print(f"📊 AVANT : {len(df.columns)} colonnes")
colonnes_avant = df.columns.tolist()


### 📁 Création du Dossier de Sortie

In [ ]:
output_folder = r"C:\Users\Y A N I S\Desktop\set\output\graphiques"
os.makedirs(output_folder, exist_ok=True)
print(f"\n📁 Dossier de sortie : {output_folder}")

# Variables numériques
numeric_cols = ['SAND', 'SILT', 'CLAY', 'BULK', 'ORG_CARBON', 'PH_WATER', 
                'CEC_SOIL', 'TOTAL_N', 'CN_RATIO']
numeric_cols = [col for col in numeric_cols if col in df.columns]

### 📈 Génération des Graphiques

#### 1️⃣ Histogrammes

In [ ]:
print("\n" + "="*70)
print("📈 GÉNÉRATION DES GRAPHIQUES (SAUVEGARDE PNG)")
print("="*70)

print("\n1️⃣ Histogrammes (9 variables)...")

n_plots = min(9, len(numeric_cols))

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Distribution des propriétés du sol', fontsize=16, fontweight='bold')

colors = ['steelblue', 'coral', 'seagreen', 'crimson', 'purple',
          'chocolate', 'pink', 'gray', 'gold']

for idx, col in enumerate(numeric_cols[:n_plots]):
    ax = axes.flat[idx]

    # 🔧 NETTOYAGE FINAL (virgule → point → float)
    data = (
        df[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
    )
    data = pd.to_numeric(data, errors='coerce').dropna()

    if data.empty:
        ax.set_visible(False)
        continue

    ax.hist(data, bins=50, color=colors[idx],
            edgecolor='black', alpha=0.7)

    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Fréquence')

    mean_val = data.mean()
    median_val = data.median()

    ax.axvline(mean_val, color='red', linestyle='--',
               linewidth=2, label=f'Moyenne: {mean_val:.2f}')
    ax.axvline(median_val, color='orange', linestyle='--',
               linewidth=2, label=f'Médiane: {median_val:.2f}')

    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Masquer les sous-graphiques inutilisés
for j in range(n_plots, 9):
    axes.flat[j].set_visible(False)

plt.tight_layout()
plt.savefig(f"{output_folder}/01_histogrammes.png",
            dpi=150, bbox_inches='tight')
plt.close()

print("   ✅ Sauvegardé : 01_histogrammes.png")


#### 2️⃣ Boxplots par Texture

In [ ]:
print("\n2️⃣ Boxplots par texture (6 variables)...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Propriétés du sol par texture USDA', fontsize=16, fontweight='bold')

vars_texture = ['PH_WATER', 'ORG_CARBON', 'CEC_SOIL', 'BULK', 'TOTAL_N', 'CN_RATIO']
vars_texture = [col for col in vars_texture if col in df.columns]

df_common = df[df['TEXTURE_SIMPLIFIED'] != 'Autres (rares)']
textures = sorted(df_common['TEXTURE_SIMPLIFIED'].unique())

for idx, col in enumerate(vars_texture[:6]):
    ax = axes[idx // 3, idx % 3]
    
    data_to_plot = [df_common[df_common['TEXTURE_SIMPLIFIED']==t][col].dropna() 
                    for t in textures]
    
    bp = ax.boxplot(data_to_plot, labels=textures, patch_artist=True)
    
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    ax.set_title(f'{col} par Texture', fontweight='bold')
    ax.set_ylabel(col)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{output_folder}/02_boxplots_texture.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 02_boxplots_texture.png")

#### 3️⃣ Boxplots par pH

In [ ]:
print("\n3️⃣ Boxplots par classe de pH (4 variables)...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Propriétés du sol par classe de pH', fontsize=16, fontweight='bold')

vars_ph = ['ORG_CARBON', 'CEC_SOIL', 'CLAY', 'SAND']
ph_classes = ['Acide (<6.5)', 'Neutre (6.5-7.5)', 'Alcalin (>7.5)']

for idx, col in enumerate(vars_ph):
    ax = axes[idx // 2, idx % 2]
    
    data_to_plot = [df[df['pH_CLASS']==pc][col].dropna() for pc in ph_classes]
    
    bp = ax.boxplot(data_to_plot, labels=ph_classes, patch_artist=True)
    
    colors_ph = ['red', 'green', 'blue']
    for patch, color in zip(bp['boxes'], colors_ph):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    
    ax.set_title(f'{col} par pH', fontweight='bold')
    ax.set_ylabel(col)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{output_folder}/03_boxplots_pH.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 03_boxplots_pH.png")

#### 4️⃣ Heatmap de Corrélation

In [ ]:
print("\n4️⃣ Heatmap de corrélation...")

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df[numeric_cols].corr()

im = ax.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=9)

ax.set_xticks(np.arange(len(numeric_cols)))
ax.set_yticks(np.arange(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
ax.set_title('Matrice de corrélation des variables du sol', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Corrélation', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig(f"{output_folder}/04_correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 04_correlation_heatmap.png")

#### 5️⃣ Scatterplots

In [ ]:
print("\n5️⃣ Scatterplots (6 relations clés)...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Relations entre variables du sol', fontsize=16, fontweight='bold')

# SAND vs CLAY
ax = axes[0, 0]
scatter = ax.scatter(df_sample['SAND'], df_sample['CLAY'], 
                     c=df_sample['SILT'], cmap='viridis', alpha=0.5, s=3)
ax.set_xlabel('SAND (%)')
ax.set_ylabel('CLAY (%)')
ax.set_title('Triangle textural (couleur = SILT)')
ax.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax, label='SILT (%)')

# ORG_CARBON vs TOTAL_N
ax = axes[0, 1]
ax.scatter(df_sample['ORG_CARBON'], df_sample['TOTAL_N'], 
           alpha=0.5, s=3, color='green')
ax.set_xlabel('ORG_CARBON (%)')
ax.set_ylabel('TOTAL_N (%)')
ax.set_title('Carbone vs Azote')
ax.grid(alpha=0.3)

# pH vs CEC
ax = axes[0, 2]
ax.scatter(df_sample['PH_WATER'], df_sample['CEC_SOIL'], 
           alpha=0.5, s=3, color='coral')
ax.set_xlabel('pH')
ax.set_ylabel('CEC_SOIL')
ax.set_title('pH vs CEC')
ax.grid(alpha=0.3)

# BULK vs ORG_CARBON
ax = axes[1, 0]
ax.scatter(df_sample['ORG_CARBON'], df_sample['BULK'], 
           alpha=0.5, s=3, color='brown')
ax.set_xlabel('ORG_CARBON (%)')
ax.set_ylabel('BULK (g/cm³)')
ax.set_title('Carbone vs Densité')
ax.grid(alpha=0.3)

# CLAY vs CEC
ax = axes[1, 1]
ax.scatter(df_sample['CLAY'], df_sample['CEC_SOIL'], 
           alpha=0.5, s=3, color='purple')
ax.set_xlabel('CLAY (%)')
ax.set_ylabel('CEC_SOIL')
ax.set_title('Argile vs CEC')
ax.grid(alpha=0.3)

# CN_RATIO
ax = axes[1, 2]
ax.scatter(df_sample['ORG_CARBON'], df_sample['CN_RATIO'], 
           alpha=0.5, s=3, color='darkgreen')
ax.set_xlabel('ORG_CARBON (%)')
ax.set_ylabel('C/N Ratio')
ax.set_title('Carbone vs Ratio C/N')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{output_folder}/05_scatterplots.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 05_scatterplots.png")

#### 6️⃣ Cartes Géographiques

In [ ]:
print("\n6️⃣ Cartes géographiques (4 propriétés)...")

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Distribution spatiale des propriétés', fontsize=16, fontweight='bold')

# pH
ax = axes[0, 0]
pivot_ph = df.pivot_table(values='PH_WATER', index='Y_grid', columns='X_grid', aggfunc='mean')
im1 = ax.imshow(pivot_ph, cmap='RdYlGn', aspect='auto', origin='lower',
                extent=[df['X'].min(), df['X'].max(), df['Y'].min(), df['Y'].max()])
ax.set_title('pH moyen par région', fontweight='bold')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
plt.colorbar(im1, ax=ax, label='pH')

# Carbone
ax = axes[0, 1]
pivot_carbon = df.pivot_table(values='ORG_CARBON', index='Y_grid', columns='X_grid', aggfunc='mean')
im2 = ax.imshow(pivot_carbon, cmap='YlGn', aspect='auto', origin='lower',
                extent=[df['X'].min(), df['X'].max(), df['Y'].min(), df['Y'].max()])
ax.set_title('Carbone organique moyen', fontweight='bold')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
plt.colorbar(im2, ax=ax, label='ORG_CARBON (%)')

# Argile
ax = axes[1, 0]
pivot_clay = df.pivot_table(values='CLAY', index='Y_grid', columns='X_grid', aggfunc='mean')
im3 = ax.imshow(pivot_clay, cmap='YlOrRd', aspect='auto', origin='lower',
                extent=[df['X'].min(), df['X'].max(), df['Y'].min(), df['Y'].max()])
ax.set_title('Argile moyenne', fontweight='bold')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
plt.colorbar(im3, ax=ax, label='CLAY (%)')

# CEC
ax = axes[1, 1]
pivot_cec = df.pivot_table(values='CEC_SOIL', index='Y_grid', columns='X_grid', aggfunc='mean')
im4 = ax.imshow(pivot_cec, cmap='plasma', aspect='auto', origin='lower',
                extent=[df['X'].min(), df['X'].max(), df['Y'].min(), df['Y'].max()])
ax.set_title('CEC moyenne', fontweight='bold')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
plt.colorbar(im4, ax=ax, label='CEC')

plt.tight_layout()
plt.savefig(f"{output_folder}/06_cartes_geographiques.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 06_cartes_geographiques.png")

#### 7️⃣ Distribution des Textures

In [ ]:
print("\n7️⃣ Distribution des textures...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribution des textures USDA', fontsize=16, fontweight='bold')

# Top 15
ax = axes[0]
texture_dist = df['TEXTURE_USDA'].value_counts().head(15)
y_pos = np.arange(len(texture_dist))
ax.barh(y_pos, texture_dist.values, color='steelblue', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(texture_dist.index, fontsize=9)
ax.set_xlabel('Nombre de pixels')
ax.set_title('Top 15 textures dominantes', fontweight='bold')
ax.grid(alpha=0.3, axis='x')

# Simplifiées
ax = axes[1]
texture_simp = df['TEXTURE_SIMPLIFIED'].value_counts()
y_pos = np.arange(len(texture_simp))
ax.barh(y_pos, texture_simp.values, color='coral', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(texture_simp.index, fontsize=9)
ax.set_xlabel('Nombre de pixels')
ax.set_title('Textures simplifiées', fontweight='bold')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f"{output_folder}/07_distribution_textures.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 07_distribution_textures.png")

#### 8️⃣ Classifications des Sols

In [ ]:
print("\n8️⃣ Distributions par classes...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Classifications des sols', fontsize=16, fontweight='bold')

# pH
ax = axes[0, 0]
ph_counts = df['pH_CLASS'].value_counts().sort_index()
ax.bar(range(len(ph_counts)), ph_counts.values, 
       color=['red', 'green', 'blue'], alpha=0.6)
ax.set_xticks(range(len(ph_counts)))
ax.set_xticklabels(ph_counts.index, rotation=0, fontsize=9)
ax.set_ylabel('Nombre de pixels')
ax.set_title('Classes de pH', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# Carbone
ax = axes[0, 1]
c_counts = df['ORG_CARBON_CLASS'].value_counts().sort_index()
ax.bar(range(len(c_counts)), c_counts.values, color='darkgreen', alpha=0.6)
ax.set_xticks(range(len(c_counts)))
ax.set_xticklabels(c_counts.index, rotation=15, ha='right', fontsize=8)
ax.set_ylabel('Nombre de pixels')
ax.set_title('Classes de carbone', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# SAND par pH
ax = axes[1, 0]
ph_classes = ['Acide (<6.5)', 'Neutre (6.5-7.5)', 'Alcalin (>7.5)']
data_sand = [df[df['pH_CLASS']==pc]['SAND'].dropna() for pc in ph_classes]
bp1 = ax.boxplot(data_sand, labels=ph_classes, patch_artist=True)
for patch, color in zip(bp1['boxes'], ['red', 'green', 'blue']):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
ax.set_ylabel('SAND (%)')
ax.set_title('Sable par pH', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# CEC par carbone
ax = axes[1, 1]
c_classes = ['Faible (<0.5%)', 'Moyen (0.5-1%)', 'Élevé (1-2%)', 'Très élevé (>2%)']
data_cec = [df[df['ORG_CARBON_CLASS']==cc]['CEC_SOIL'].dropna() for cc in c_classes]
bp2 = ax.boxplot(data_cec, labels=c_classes, patch_artist=True)
for patch in bp2['boxes']:
    patch.set_facecolor('lightgreen')
    patch.set_alpha(0.6)
ax.set_ylabel('CEC_SOIL')
ax.set_title('CEC par carbone', fontweight='bold')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right', fontsize=8)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{output_folder}/08_classes_pH_carbone.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"   ✅ Sauvegardé : 08_classes_pH_carbone.png")

---
## Partie 2: Diagnostic des Lignes Suspectes

### 🔍 Analyse des Valeurs Manquantes

In [ ]:
print("\n" + "="*70)
print("🔍 DIAGNOSTIC DES LIGNES SUSPECTES")
print("="*70)

# Charger le CSV brut pour diagnostic
csv_path = r"C:\Users\Y A N I S\output\HWSD2_Algeria_Tunisia_D1_RAW.csv"
df_raw = pd.read_csv(csv_path, sep=';', decimal=',', encoding='utf-8-sig')

print(f"\n📊 Dataset initial : {len(df_raw):,} lignes × {len(df_raw.columns)} colonnes")

### 1️⃣ Identifier les Lignes avec Valeurs Manquantes

In [ ]:
print("\n" + "="*70)
print("1️⃣ LIGNES AVEC VALEURS MANQUANTES (TEXTURE_USDA ou REF_BULK)")
print("="*70)

mask_missing = df_raw['TEXTURE_USDA'].isna() | df_raw['REF_BULK'].isna()
df_missing = df_raw[mask_missing].copy()

print(f"\n🔴 {len(df_missing):,} lignes ont TEXTURE_USDA ou REF_BULK manquant")
print(f"   ({len(df_missing)/len(df_raw)*100:.2f}% du dataset)")

### 2️⃣ Analyse des Valeurs dans ces Lignes

In [ ]:
print("\n" + "="*70)
print("2️⃣ ANALYSE DES VALEURS DANS CES LIGNES SUSPECTES")
print("="*70)

numeric_cols_diag = ['COARSE', 'SAND', 'SILT', 'CLAY', 'BULK', 'ORG_CARBON', 
                     'PH_WATER', 'TOTAL_N', 'CN_RATIO', 'CEC_SOIL']
numeric_cols_diag = [col for col in numeric_cols_diag if col in df_raw.columns]

print(f"\n📊 Aperçu des 10 premières lignes suspectes :")
print(df_missing[['HWSD2_SMU_ID', 'TEXTURE_USDA', 'REF_BULK'] + numeric_cols_diag[:5]].head(10))

### 3️⃣ Détection des Duplicates Complets

In [ ]:
print("\n" + "="*70)
print("3️⃣ DÉTECTION DES DUPLICATES COMPLETS (hors X,Y)")
print("="*70)

cols_to_check = [col for col in df_raw.columns if col not in ['X', 'Y']]
df_missing['is_duplicate'] = df_missing.duplicated(subset=cols_to_check, keep=False)
n_duplicates = df_missing['is_duplicate'].sum()

print(f"\n🔴 {n_duplicates:,} lignes suspectes sont des duplicates COMPLETS")

### 4️⃣ Lignes avec Valeurs Identiques

In [ ]:
print("\n" + "="*70)
print("4️⃣ LIGNES OÙ TOUTES LES VALEURS NUMÉRIQUES SONT IDENTIQUES")
print("="*70)

def all_values_same(row):
    values = [row[col] for col in numeric_cols_diag if pd.notna(row[col])]
    if len(values) <= 1:
        return False
    return len(set(values)) == 1

df_missing['all_same'] = df_missing.apply(all_values_same, axis=1)
n_all_same = df_missing['all_same'].sum()

print(f"\n🔴 {n_all_same:,} lignes où toutes les valeurs numériques sont identiques")

### 5️⃣ Valeurs Aberrantes

In [ ]:
print("\n" + "="*70)
print("5️⃣ VALEURS ABERRANTES (0, -999, etc.)")
print("="*70)

df_missing['n_zeros'] = (df_missing[numeric_cols_diag] == 0).sum(axis=1)
df_missing['n_negative'] = (df_missing[numeric_cols_diag] < 0).sum(axis=1)

print(f"\n📊 Distribution des zéros dans les lignes suspectes :")
print(df_missing['n_zeros'].value_counts().sort_index())

n_many_zeros = (df_missing['n_zeros'] >= len(numeric_cols_diag) * 0.5).sum()
print(f"\n🔴 {n_many_zeros:,} lignes avec >50% de zéros")

### 6️⃣ Comparaison avec Lignes Normales

In [ ]:
print("\n" + "="*70)
print("6️⃣ COMPARAISON AVEC LES LIGNES NORMALES")
print("="*70)

df_normal = df_raw[~mask_missing]

print(f"\n✅ Lignes NORMALES ({len(df_normal):,}) :")
print(df_normal[numeric_cols_diag].describe().loc[['mean', 'std', 'min', 'max']])

print(f"\n🔴 Lignes SUSPECTES ({len(df_missing):,}) :")
print(df_missing[numeric_cols_diag].describe().loc[['mean', 'std', 'min', 'max']])

### 7️⃣ Export des Lignes Suspectes

In [ ]:
output_suspect = r"C:\Users\Y A N I S\Desktop\set\output\HWSD2_lignes_suspectes.csv"
df_missing.to_csv(output_suspect, index=False, sep=';', decimal=',', encoding='utf-8-sig')
print(f"✅ Lignes suspectes exportées : {output_suspect}")

---
## Partie 3: Traitement Final - Capping, Agrégation et Encodage

### 🔧 Définition des Variables

In [ ]:
print("\n" + "="*70)
print("🔧 TRAITEMENT FINAL DES DONNÉES")
print("="*70)

# Définir les colonnes numériques et catégorielles
numeric_vars = ['COARSE', 'SAND', 'SILT', 'CLAY', 'BULK', 'ORG_CARBON', 'PH_WATER', 
                'TOTAL_N', 'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 
                'BSAT', 'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'REF_BULK']
numeric_vars = [col for col in numeric_vars if col in df.columns]

categorical_vars = ['TEXTURE_USDA', 'TEXTURE_SOTER']
categorical_vars = [col for col in categorical_vars if col in df.columns]

print(f"\n✅ Variables numériques ({len(numeric_vars)}) : {numeric_vars[:5]}...")
print(f"✅ Variables catégorielles ({len(categorical_vars)}) : {categorical_vars}")

### 1️⃣ Gestion des Valeurs Aberrantes (Capping 1%-99%)

In [ ]:
print("\n" + "="*70)
print("1️⃣ CAPPING DES VALEURS ABERRANTES (1%-99%)")
print("="*70)

df_capped = df.copy()

for col in numeric_vars:
    if col in df_capped.columns:
        q01 = df_capped[col].quantile(0.01)
        q99 = df_capped[col].quantile(0.99)
        
        # Nombre de valeurs hors limites
        n_outliers = ((df_capped[col] < q01) | (df_capped[col] > q99)).sum()
        
        # Application du capping
        df_capped[col] = df_capped[col].clip(lower=q01, upper=q99)
        
        print(f"   • {col:15s} : Q1%={q01:8.2f}, Q99%={q99:8.2f}, Outliers={n_outliers:6,}")

print(f"\n✅ Capping appliqué sur {len(numeric_vars)} variables")

### 2️⃣ Agrégation par Pixel (X, Y)

In [ ]:
print("\n" + "="*70)
print("2️⃣ AGRÉGATION PAR PIXEL (X, Y)")
print("="*70)

# Fonction de mode robuste pour les variables catégorielles
def mode_robuste(serie):
    """Retourne la valeur la plus fréquente, ou la première valeur non-NaN si pas de mode"""
    serie_clean = serie.dropna()
    if len(serie_clean) == 0:
        return np.nan
    mode_result = serie_clean.mode()
    if len(mode_result) > 0:
        return mode_result.iloc[0]
    return serie_clean.iloc[0]

# Dictionnaire des fonctions d'agrégation
agg_functions = {}

# Variables numériques : moyenne
for col in numeric_vars:
    if col in df_capped.columns:
        agg_functions[col] = 'mean'

# Variables catégorielles : mode robuste
for col in categorical_vars:
    if col in df_capped.columns:
        agg_functions[col] = mode_robuste

print(f"\n📊 Avant agrégation : {len(df_capped):,} lignes")

# Agrégation
df_aggregated = df_capped.groupby(['X', 'Y']).agg(agg_functions).reset_index()

print(f"📊 Après agrégation : {len(df_aggregated):,} pixels uniques")
print(f"📉 Réduction : {len(df_capped) - len(df_aggregated):,} lignes ({(1 - len(df_aggregated)/len(df_capped))*100:.1f}%)")

### 3️⃣ Encodage One-Hot des Variables Catégorielles

In [ ]:
print("\n" + "="*70)
print("3️⃣ ONE-HOT ENCODING DES VARIABLES CATÉGORIELLES")
print("="*70)

df_encoded = df_aggregated.copy()

# Encodage pour TEXTURE_SOTER
if 'TEXTURE_SOTER' in df_encoded.columns:
    texture_soter_dummies = pd.get_dummies(df_encoded['TEXTURE_SOTER'], prefix='Texture_soter')
    df_encoded = pd.concat([df_encoded, texture_soter_dummies], axis=1)
    print(f"✅ TEXTURE_SOTER encodée : {list(texture_soter_dummies.columns)}")
    df_encoded = df_encoded.drop('TEXTURE_SOTER', axis=1)

# Encodage pour TEXTURE_USDA
if 'TEXTURE_USDA' in df_encoded.columns:
    texture_usda_dummies = pd.get_dummies(df_encoded['TEXTURE_USDA'], prefix='TEXTURE_USDA')
    df_encoded = pd.concat([df_encoded, texture_usda_dummies], axis=1)
    print(f"✅ TEXTURE_USDA encodée : {len(texture_usda_dummies.columns)} colonnes créées")
    print(f"   Exemples : {list(texture_usda_dummies.columns[:5])}")
    df_encoded = df_encoded.drop('TEXTURE_USDA', axis=1)

print(f"\n📊 Dataset final : {len(df_encoded):,} lignes × {len(df_encoded.columns)} colonnes")

### 4️⃣ Export du Dataset Final

In [ ]:
print("\n" + "="*70)
print("4️⃣ EXPORT DU DATASET FINAL")
print("="*70)

# Export CSV
output_final = r"C:\Users\Y A N I S\Desktop\set\output\HWSD2_Algeria_Tunisia_FINAL.csv"
df_encoded.to_csv(output_final, index=False, sep=';', decimal=',', encoding='utf-8-sig')

print(f"🎉 Dataset final exporté : {output_final}")
print(f"\n📋 Résumé :")
print(f"   • Lignes : {len(df_encoded):,}")
print(f"   • Colonnes : {len(df_encoded.columns)}")
print(f"   • Coordonnées : X, Y")
print(f"   • Variables numériques cappées : {len(numeric_vars)}")
print(f"   • Variables catégorielles encodées : {len(categorical_vars)}")

# Afficher les 5 premières lignes
print(f"\n📊 Aperçu du dataset final :")
print(df_encoded.head())

### ✅ Résumé Complet

In [ ]:
print("\n" + "="*70)
print("✅ TRAITEMENT COMPLET TERMINÉ")
print("="*70)

print(f"""
📊 RÉCAPITULATIF DU TRAITEMENT :

1️⃣ CAPPING DES VALEURS ABERRANTES
   • Percentiles 1%-99% appliqués sur {len(numeric_vars)} variables
   • Valeurs extrêmes limitées pour éviter les biais

2️⃣ AGRÉGATION SPATIALE PAR PIXEL
   • Variables numériques : moyenne par pixel (X, Y)
   • Variables catégorielles : mode robuste (valeur la plus fréquente)
   • Réduction : {len(df_capped):,} → {len(df_aggregated):,} pixels

3️⃣ ENCODAGE ONE-HOT
   • TEXTURE_SOTER : {len([c for c in df_encoded.columns if 'Texture_soter' in c])} colonnes binaires
   • TEXTURE_USDA : {len([c for c in df_encoded.columns if 'TEXTURE_USDA' in c])} colonnes binaires

📁 FICHIERS GÉNÉRÉS :
   • {output_final}
   • {output_suspect}
   • {output_folder}/ (graphiques PNG)

💡 COLONNES ONE-HOT CRÉÉES :
   • Texture_soter_C, Texture_soter_F, Texture_soter_M
   • TEXTURE_USDA_3.0, TEXTURE_USDA_5.0, TEXTURE_USDA_7.0, etc.
""")